### Getting Started 

# Welcome to the Orbit of Ops Workspace

Hello! This notebook is prepared for processing sensitive data with custom branding. Please follow the instructions below to proceed with your tasks.

#### Install Google Gen AI SDK for Python

In [1]:
%pip install --upgrade --quiet google-genai

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.130.0 requires google-genai<2.0.0,>=1.37.0, but you have google-genai 2.14.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


#### Restart runtime
To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

In [ ]:
# ==============================================================================
# 🚀 ORBIT OF OPS HORIZON APP 
# 🎯 MISSION: GSP515 - Task 2 (Kernel Repair Protocol)
# ==============================================================================
!pip install --upgrade --quiet --force-reinstall google-auth google-genai google-cloud-aiplatform
import os
print("\n# ==============================================================================")
print("# ✅ REPAIR COMPLETE. The kernel will now restart automatically.")
print("# Wait 5 seconds for the status to say 'Idle' before running Task 3!")
print("# ==============================================================================")
os._exit(0)

In [ ]:
# restart the kernel after libraries are loaded
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

### Set Google Cloud project information and create client

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).



In [ ]:
# Define project information
PROJECT_ID = "qwiklabs-gcp-03-3b21a3fc69a5"  # @param {type:"string"}
LOCATION = "global"

# Create the API client
from google import genai
client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

#### Import libraries


In [ ]:
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    Tool,
    Part
)

### Task 3. Create a function call using Gemini

In [ ]:
# Task 3.1
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
# Load Gemini 3.5 Flash Model
model_id = "gemini-3.5-flash"

In [ ]:
# Task 3.2
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
get_current_weather_func = FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather in a given location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "Location"
            }
        }
    },
)

In [ ]:
# Task 3.3
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
weather_tool = Tool(
    function_declarations=[get_current_weather_func],
)

In [ ]:
# Task 3.4
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
prompt = "What is the weather like in Boston?"
response = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=GenerateContentConfig(
        tools=[weather_tool],
        temperature=0,
    ),
)
response


[!] Orbit of Ops Command Center: Initializing Task 3...
[*] Authenticating with Project ID: qwiklabs-gcp-03-3b21a3fc69a5 in Multi-Region: us
[*] Executing Gemini Function Call...

✅ Function Call Generated Successfully:
id='oone9ttq' args={'location': 'Boston'} name='get_current_weather' partial_args=None will_continue=None

# ==============================================================================
# ✅ TASK 3 COMPLETE! Save your notebook and click 'Check my progress'.
# ==============================================================================


### Task 4. Describe video contents using Gemini

In [ ]:
# Run the following cell to import required libraries 
from google.genai.types import (
    GenerationConfig,
    Image,
    Part,
)

In [ ]:
# Task 4.1
# Load the correct Gemini model use the following documentation to assist:
# https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/overview#supported-use-cases
# Load Gemini 3.5 Flash Model
multimodal_model = "gemini-3.5-flash"

In [ ]:
import http.client
import typing
import urllib.request

import IPython.display
from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps


def display_images(
    images: typing.Iterable[Image],
    max_width: int = 600,
    max_height: int = 350,
) -> None:
    for image in images:
        pil_image = typing.cast(PIL_Image.Image, image._pil_image)
        if pil_image.mode != "RGB":
            # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
            pil_image = pil_image.convert("RGB")
        image_width, image_height = pil_image.size
        if max_width < image_width or max_height < image_height:
            # Resize to display a smaller notebook image
            pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
        IPython.display.display(pil_image)


def get_image_bytes_from_url(image_url: str) -> bytes:
    with urllib.request.urlopen(image_url) as response:
        response = typing.cast(http.client.HTTPResponse, response)
        image_bytes = response.read()
    return image_bytes


def load_image_from_url(image_url: str) -> Image:
    image_bytes = get_image_bytes_from_url(image_url)
    return Image.from_bytes(image_bytes)


def display_content_as_image(content: str | Image | Part) -> bool:
    if not isinstance(content, Image):
        return False
    display_images([content])
    return True


def display_content_as_video(content: str | Image | Part) -> bool:
    if not isinstance(content, Part):
        return False
    part = typing.cast(Part, content)
    file_path = part.file_data.file_uri.removeprefix("gs://")
    video_url = f"https://storage.googleapis.com/{file_path}"
    IPython.display.display(IPython.display.Video(video_url, width=600))
    return True


def print_multimodal_prompt(contents: list[str | Image | Part]):
    """
    Given contents that would be sent to Gemini,
    output the full multimodal prompt for ease of readability.
    """
    for content in contents:
        if display_content_as_image(content):
            continue
        if display_content_as_video(content):
            continue
        print(content)

In [ ]:
# Task 4.2 Generate a video description
# In this cell, update the prompt to ask Gemini to describe the video URL referenced.
# You can use the documentation at the following link to assist.
# https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/sdk-for-gemini/gemini-sdk-overview-reference#generate-content-from-video
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference#sample-requests-text-stream-response
# Video URI: gs://github-repo/img/gemini/multimodality_usecases_overview/mediterraneansea.mp4

prompt = """
What is shown in this video?
Where should I go to see it?
What are the top 5 places in the world that look like this?
"""
video = Part.from_uri(
    file_uri="gs://github-repo/img/gemini/multimodality_usecases_overview/mediterraneansea.mp4",
    mime_type="video/mp4",
)
contents = [prompt, video]

responses = client.models.generate_content_stream(
    model=multimodal_model,
    contents=contents
)

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")


[!] Orbit of Ops Command Center: Initializing Task 4...
[*] Authenticating with Project ID: qwiklabs-gcp-03-3b21a3fc69a5 in Multi-Region: us
[*] Executing Gemini Video Analysis (Streaming)...

✅ Video Analysis Generated:



In [ ]:
import base64
from IPython.display import display, HTML

# 1. Read the local image file and encode it
with open("logo.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode('utf-8')

# 2. Create the data URI for the HTML src attribute
b64_image_data = f"data:image/png;base64,{encoded_string}"

# 3. Inject it directly into the HTML
display(HTML(f'''
<div style="text-align: center; padding: 25px; background-color: #ffffff; border: 2px solid #1a73e8; border-radius: 12px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <img src="{b64_image_data}" width="250" />
    <h1 style="color: #1a73e8; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin-top: 15px;">Orbit of Ops</h1>
    <p style="color: #5f6368; font-size: 1.1em;"><i>Optimizing Operations via Intelligence</i></p>
    <hr style="width: 50%; border: 0; border-top: 1px solid #eee; margin: 15px auto;">
    <a href="https://orbitofops.com" style="color: #1a73e8; text-decoration: none; font-weight: bold;">orbitofops.com</a>
</div>
'''))